In [0]:
# ── Cell 1: Load bronze snapshot ─────────────────────────────────────────
storage_account = "onlinelearningd1"
key = dbutils.secrets.get(scope="adls-scope", key="storage-key")
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", key)

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window

df = spark.read.parquet(
    f"abfss://silver@{storage_account}.dfs.core.windows.net/bronze_snapshot/"
)
print(f"Loaded {df.count()} rows")

Loaded 1225 rows


In [0]:
# ── Cell 2: Step 1 — Remove exact duplicate rows ─────────────────────────
before = df.count()
df = df.dropDuplicates()
print(f"Step 1 — Deduplication: removed {before - df.count()} rows. Remaining: {df.count()}")

Step 1 — Deduplication: removed 21 rows. Remaining: 1204


In [0]:
# ── Cell 3: Step 2 — Strip whitespace from all string columns ─────────────
str_cols = ["user_id","gender","country","education_level",
            "preferred_device","course_completed"]
for col in str_cols:
    df = df.withColumn(col, F.trim(F.col(col)))
print("Step 2 — Whitespace stripped.")

Step 2 — Whitespace stripped.


In [0]:
# ── Cell 4: Step 3 — Standardise gender ──────────────────────────────────
df = df.withColumn("gender",
    F.when(F.lower(F.col("gender")).isin(["male","m","man"]),      "Male")
     .when(F.lower(F.col("gender")).isin(["female","f","woman"]),  "Female")
     .when(F.lower(F.col("gender")).isin(["other","others","non-binary","non_binary"]), "Other")
     .otherwise(F.col("gender"))
)
print("Step 3 — Gender standardised:", [r[0] for r in df.select("gender").distinct().collect()])

Step 3 — Gender standardised: ['Female', 'Other', 'Male']


In [0]:
# ── Cell 5: Step 4 — Standardise education_level ─────────────────────────
df = df.withColumn("education_level",
    F.when(F.lower(F.trim(F.col("education_level"))).isin(
           ["bachelor","bachelors","b.tech","b.sc"]),          "Bachelor")
     .when(F.lower(F.trim(F.col("education_level"))).isin(
           ["master","masters","m.tech","msc","m.s"]),          "Master")
     .when(F.lower(F.trim(F.col("education_level"))).isin(
           ["phd","ph.d","doctorate","p.h.d"]),                 "PhD")
     .when(F.lower(F.trim(F.col("education_level"))).isin(
           ["high school","highschool"]),                       "High School")
     .otherwise(F.col("education_level"))
)
print("Step 4 — Education standardised:", [r[0] for r in df.select("education_level").distinct().collect()])


Step 4 — Education standardised: ['High School', 'PhD', 'Other', 'Master', 'Bachelor']


In [0]:
# ── Cell 6: Step 5 — Standardise preferred_device ────────────────────────
df = df.withColumn("preferred_device",
    F.when(F.lower(F.col("preferred_device")).isin(
           ["mobile","phone","smartphone"]),          "Mobile")
     .when(F.lower(F.col("preferred_device")).isin(
           ["desktop","pc","laptop"]),                "Desktop")
     .when(F.lower(F.col("preferred_device")).isin(
           ["tablet","ipad","tab"]),                  "Tablet")
     .otherwise(F.col("preferred_device"))
)


In [0]:
# ── Cell 7: Step 6 — Standardise course_completed ────────────────────────
df = df.withColumn("course_completed",
    F.when(F.lower(F.col("course_completed")).isin(["yes","1","true"]),  "Yes")
     .when(F.lower(F.col("course_completed")).isin(["no","0","false"]),  "No")
     .otherwise(F.col("course_completed"))
)
print("Step 6 — Course completed unique:", [r[0] for r in df.select("course_completed").distinct().collect()])


Step 6 — Course completed unique: ['No', 'Yes']


In [0]:
# ── Cell 8: Step 7 — Cast to correct data types ───────────────────────────
df = (df
    .withColumn("age",                 F.col("age").cast(DoubleType()).cast(IntegerType()))
    .withColumn("hours_spent_weekly",  F.col("hours_spent_weekly").cast(DoubleType()))
    .withColumn("satisfaction_rating", F.col("satisfaction_rating").cast(IntegerType()))
    .withColumn("login_frequency",     F.col("login_frequency").cast(IntegerType()))
    .withColumn("quiz_score_avg",      F.col("quiz_score_avg").cast(DoubleType()))
)
print("Step 7 — Dtypes cast.")
df.printSchema()

Step 7 — Dtypes cast.
root
 |-- user_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- hours_spent_weekly: double (nullable = true)
 |-- preferred_device: string (nullable = true)
 |-- course_completed: string (nullable = true)
 |-- satisfaction_rating: integer (nullable = true)
 |-- login_frequency: integer (nullable = true)
 |-- quiz_score_avg: double (nullable = true)



In [0]:
# ── Cell 9: Step 8 — Remove outliers (domain-bound filtering) ─────────────
before_outlier = df.count()
df = df.filter(
    (F.col("age").isNull()                | F.col("age").between(10, 100))        &
    (F.col("hours_spent_weekly").isNull() | F.col("hours_spent_weekly").between(0, 168)) &
    (F.col("quiz_score_avg").isNull()     | F.col("quiz_score_avg").between(0, 100))    &
    (F.col("satisfaction_rating").isNull()| F.col("satisfaction_rating").between(1, 5)) &
    (F.col("login_frequency").isNull()    | F.col("login_frequency").between(0, 30))
)
print(f"Step 8 — Outliers removed: {before_outlier - df.count()} rows. Remaining: {df.count()}")


Step 8 — Outliers removed: 41 rows. Remaining: 1163


In [0]:
# ── Cell 10: Step 9 — Impute missing values ───────────────────────────────
hours_med = df.approxQuantile("hours_spent_weekly",  [0.5], 0.01)[0]
quiz_med  = df.approxQuantile("quiz_score_avg",      [0.5], 0.01)[0]
sat_med   = int(df.approxQuantile("satisfaction_rating", [0.5], 0.01)[0])
login_med = int(df.approxQuantile("login_frequency", [0.5], 0.01)[0])

df = (df
    .fillna({"hours_spent_weekly":  round(hours_med, 1)})
    .fillna({"quiz_score_avg":      round(quiz_med,  1)})
    .fillna({"satisfaction_rating": sat_med})
    .fillna({"login_frequency":     login_med})
    .fillna({"country":             "Unknown"})
)

# Verify no nulls remain
null_check = {c: df.filter(F.col(c).isNull()).count() for c in df.columns}
print(f"Step 9 — Nulls after imputation: {null_check}")


Step 9 — Nulls after imputation: {'user_id': 0, 'age': 0, 'gender': 0, 'country': 0, 'education_level': 0, 'hours_spent_weekly': 0, 'preferred_device': 0, 'course_completed': 0, 'satisfaction_rating': 0, 'login_frequency': 0, 'quiz_score_avg': 0}


In [0]:
# ── Cell 11: Step 10 — Feature engineering ────────────────────────────────
df = (df
    .withColumn("engagement_score",
        F.round(F.col("login_frequency") * 0.5 + F.col("hours_spent_weekly") * 0.5, 2))
    .withColumn("satisfaction_group",
        F.when(F.col("satisfaction_rating") <= 2, "Low")
         .when(F.col("satisfaction_rating") == 3, "Medium")
         .otherwise("High"))
    .withColumn("completed_binary",
        F.when(F.col("course_completed") == "Yes", 1).otherwise(0))
    .withColumn("study_band",
        F.when(F.col("hours_spent_weekly") < 5,  "0-5 hrs")
         .when(F.col("hours_spent_weekly") < 10, "5-10 hrs")
         .when(F.col("hours_spent_weekly") < 15, "10-15 hrs")
         .otherwise("15+ hrs"))
    .withColumn("processed_at", F.current_timestamp())
)
print("Step 10 — Feature engineering done.")

Step 10 — Feature engineering done.


In [0]:
# ── Cell 12: Write Silver Parquet ─────────────────────────────────────────
(df.write
    .partitionBy("education_level")
    .mode("overwrite")
    .parquet(f"abfss://silver@{storage_account}.dfs.core.windows.net/clean/")
)
print(f"Silver zone written. Final row count: {df.count()}")
display(df.limit(5))

Silver zone written. Final row count: 1163


user_id,age,gender,country,education_level,hours_spent_weekly,preferred_device,course_completed,satisfaction_rating,login_frequency,quiz_score_avg,engagement_score,satisfaction_group,completed_binary,study_band,processed_at
user_0468,19,Other,Australia,Bachelor,9.6,Desktop,No,5,2,67.9,5.8,High,0,5-10 hrs,2026-04-11T14:13:49.056Z
user_0723,44,Female,Australia,Master,6.2,Mobile,No,3,9,57.1,7.6,Medium,0,5-10 hrs,2026-04-11T14:13:49.056Z
user_0118,20,Other,Canada,PhD,5.1,Mobile,No,2,1,67.0,3.05,Low,0,5-10 hrs,2026-04-11T14:13:49.056Z
user_0781,26,Male,India,Master,7.9,Tablet,No,3,12,56.5,9.95,Medium,0,5-10 hrs,2026-04-11T14:13:49.056Z
user_0340,57,Other,Australia,Master,8.3,Desktop,Yes,3,2,94.5,5.15,Medium,1,5-10 hrs,2026-04-11T14:13:49.056Z
